In [1]:
import pandas as pd

In [7]:
df = pd.read_csv('Multiple Cause of Death, 1999-2014.csv')
df.head() 

,State,Year,Deaths,Population,Crude Rate,Crude Rate Lower 95% Confidence Interval,Crude Rate Upper 95% Confidence Interval
0,Alabama,1999,39,4430141,0.9,0.6,1.2
1,Alabama,2000,46,4447100,1,0.8,1.4
2,Alabama,2001,67,4467634,1.5,1.2,1.9
3,Alabama,2002,75,4480089,1.7,1.3,2.1
4,Alabama,2003,54,4503491,1.2,0.9,1.6


In [108]:
df.columns = [col.replace(' ', '_') for col in df.columns]
df.head()

,State,Year,Deaths,Population,Crude_Rate,Crude_Rate_Lower_95%_Confidence_Interval,Crude_Rate_Upper_95%_Confidence_Interval
0,Alabama,1999,39,4430141,0.9,0.6,1.2
1,Alabama,2000,46,4447100,1,0.8,1.4
2,Alabama,2001,67,4467634,1.5,1.2,1.9
3,Alabama,2002,75,4480089,1.7,1.3,2.1
4,Alabama,2003,54,4503491,1.2,0.9,1.6


In [110]:
df.dtypes

State                                       object
Year                                         int64
Deaths                                      object
Population                                   int64
Crude_Rate                                  object
Crude_Rate_Lower_95%_Confidence_Interval    object
Crude_Rate_Upper_95%_Confidence_Interval    object
dtype: object

In [114]:
#remove rows where Deaths are Suppressed and CrudeRate are marked as unreliable
df_cleaned = df.loc[(df['Deaths'] != 'Suppressed') & (df['Crude_Rate'] != 'Unreliable')].copy()
df_cleaned

,State,Year,Deaths,Population,Crude_Rate,Crude_Rate_Lower_95%_Confidence_Interval,Crude_Rate_Upper_95%_Confidence_Interval
0,Alabama,1999,39,4430141,0.9,0.6,1.2
1,Alabama,2000,46,4447100,1,0.8,1.4
2,Alabama,2001,67,4467634,1.5,1.2,1.9
3,Alabama,2002,75,4480089,1.7,1.3,2.1
4,Alabama,2003,54,4503491,1.2,0.9,1.6
...,...,...,...,...,...,...,...
811,Wyoming,2010,49,563626,8.7,6.4,11.5
812,Wyoming,2011,47,568158,8.3,6.1,11
813,Wyoming,2012,47,576412,8.2,6,10.8
814,Wyoming,2013,52,582658,8.9,6.7,11.7


In [118]:
#clean data types of Death and Crude Rate
df_cleaned['Deaths'] = pd.to_numeric(df_cleaned['Deaths'])
df_cleaned['Crude_Rate'] = pd.to_numeric(df_cleaned['Crude_Rate'])

df_cleaned.dtypes

State                                        object
Year                                          int64
Deaths                                        int64
Population                                    int64
Crude_Rate                                  float64
Crude_Rate_Lower_95%_Confidence_Interval     object
Crude_Rate_Upper_95%_Confidence_Interval     object
dtype: object

In [120]:
#query find all states & years where number of deaths >2000 per year
df_cleaned.query('Deaths>2000')[['State','Year']]

,State,Year
74,California,2009
75,California,2010
76,California,2011
78,California,2013
79,California,2014
575,Ohio,2014


In [122]:
df_summary = df_cleaned.groupby('State')['Deaths'].sum().reset_index(name='Total')
df_summary.head()

,State,Total
0,Alabama,2173
1,Alaska,644
2,Arizona,7298
3,Arkansas,2178
4,California,27044


In [126]:
maskA = df_cleaned['State'].str[0] == 'A'

df_states_with_A = df_cleaned.iloc[maskA.index[mask]]
df_states_with_A

IndexError: boolean index did not match indexed array along dimension 0; dimension is 772 but corresponding boolean dimension is 816

In [128]:
df_population = df_cleaned.groupby('State')['Population'].max().reset_index(name='Population')
df_top_five_states = df_population.sort_values(by='Population', ascending=False).head(5)
df_top_five_states

,State,Population
4,California,38802500
43,Texas,26956958
9,Florida,19893297
32,New York,19746227
13,Illinois,12882135


In [130]:
top_population_state = df_top_five_states.iloc[0]['State']
df_state = df_cleaned[df_cleaned['State'] == top_population_state]
df_top_five_years = df_state.nlargest(5, 'Deaths')[['Year', 'Deaths']]
df_top_five_years

,Year,Deaths
79,2014,2159
74,2009,2128
78,2013,2088
75,2010,2059
76,2011,2057


### Meaningful Question
#### What is the average number of deaths per state in order of greatest population to smallest?

In [142]:
df_avg_deaths_per_state = df_cleaned.groupby('State')['Deaths'].mean().reset_index(name='Average_Deaths')
df_population = df_cleaned.groupby('State')['Population'].max().reset_index(name='Population')
df_avg_deaths_with_population = pd.merge(df_avg_deaths_per_state, df_population,on ='State')
df_sorted = df_avg_deaths_with_population.sort_values('Population', ascending=False)
df_sorted[['State', 'Average_Deaths','Population']].head()

,State,Average_Deaths,Population
4,California,1690.2500,38802500
43,Texas,940.6250,26956958
9,Florida,1244.9375,19893297
32,New York,1009.7500,19746227
13,Illinois,817.0000,12882135
